# Lab 3 — ROCm Libraries, PyTorch & AI Frameworks

**ROCm Certification Program — Level 1**

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Objectives</div>
<p>Most AI, signal-processing, and scientific workloads reduce to a few primitives — matrix multiply, convolution, FFT. In this lab you will:</p><ul style='margin-bottom:0;'><li>Call <b>rocBLAS</b>, <b>MIOpen</b>, and <b>rocFFT</b> directly and measure performance</li><li>Run an end-to-end <b>PyTorch</b> training loop on an AMD GPU</li><li>Diagnose and fix common ROCm + PyTorch failures</li></ul>
</div>

### Libraries used in this lab

| Library | Domain | What it provides | Used in |
|---------|--------|------------------|---------|
| **rocBLAS** | Linear algebra | BLAS routines, matrix multiply (GEMM) | Exercise 1 |
| **MIOpen** | Deep learning | Convolutions, pooling, activations | Exercise 2 |
| **rocFFT** | Signal processing | Fast Fourier Transforms | Exercise 3 |
| **PyTorch** | AI framework | High-level ML built on the libraries above | Exercises 4–5 |

## Setup — Verify the ROCm + PyTorch Environment

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Goal</div>
Confirm Python, PyTorch, and ROCm/HIP are installed and that PyTorch can see the AMD GPU before running any exercise.
</div>

In [1]:
import sys

print(sys.version)
print(sys.executable)

import torch

print(torch.__version__)
print(torch.version.hip)

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
/opt/venv/bin/python
2.10.0+rocm7.2.4.git3d3aa833
7.2.53211
True
AMD Radeon Graphics


In [2]:
# Verify ROCm PyTorch build
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"HIP available:   {torch.cuda.is_available()}")
print(f"HIP version:     {torch.version.hip}")
print(f"GPU:             {torch.cuda.get_device_name(0)}")
print(f"GPU memory:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.10.0+rocm7.2.4.git3d3aa833
HIP available:   True
HIP version:     7.2.53211
GPU:             AMD Radeon Graphics
GPU memory:      51.5 GB


<div style="background:#f4f6f9; border-left:5px solid #64748b; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#334155; font-size:1.05em; margin-bottom:8px;">Expected result</div>
<p>A correctly configured environment reports a Python version, the PyTorch version (e.g. <code>2.10.0+rocm7.2.4</code>), a ROCm/HIP version, <code>True</code> for GPU availability, and the GPU name (e.g. <code>AMD Radeon Graphics</code>). If any check fails, fix the ROCm install, PyTorch wheel, or driver before continuing.</p>
</div>

## Exercise 1 — rocBLAS GEMM

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Goal</div>
<p><b>rocBLAS</b> is AMD's optimized BLAS implementation for ROCm GPUs. Compile and run a rocBLAS program that calls <code>rocblas_sgemm()</code> for single-precision matrix multiply, then measure throughput in GFLOPS.</p>
</div>

In [3]:
%%writefile rocblas_gemm.cpp
#include <hip/hip_runtime.h>
#include <rocblas/rocblas.h>
#include <cstdio>
#include <cstdlib>
#include <cmath>

int main() {

    // Matrix dimensions:
    // A = M x K
    // B = K x N
    // C = M x N
    //
    // We will compute:
    //
    //     C = alpha * A * B + beta * C
    //
    // This operation is known as GEMM
    // (General Matrix Multiply).
    //
    int M = 1024, N = 1024, K = 1024;

    float alpha = 1.0f, beta = 0.1f;

    // Calculate buffer sizes in bytes
    size_t sA = M*K*sizeof(float);
    size_t sB = K*N*sizeof(float);
    size_t sC = M*N*sizeof(float);

    // Allocate host (CPU) memory
    float *h_A=(float*)malloc(sA);
    float *h_B=(float*)malloc(sB);
    float *h_C=(float*)malloc(sC);

    // Generate reproducible random input data
    srand(42);

    for(int i=0;i<M*K;i++)
        h_A[i]=(float)rand()/RAND_MAX;

    for(int i=0;i<K*N;i++)
        h_B[i]=(float)rand()/RAND_MAX;

    for(int i=0;i<K*N;i++)
        h_C[i]=(float)rand()/RAND_MAX;

    // Allocate device (GPU) memory
    float *d_A,*d_B,*d_C;

    hipMalloc(&d_A,sA);
    hipMalloc(&d_B,sB);
    hipMalloc(&d_C,sC);

    // Copy matrices from CPU memory to GPU memory
    hipMemcpy(d_A,h_A,sA,hipMemcpyHostToDevice);
    hipMemcpy(d_B,h_B,sB,hipMemcpyHostToDevice);
    hipMemcpy(d_C,h_C,sC,hipMemcpyHostToDevice);

    //
    // Create a rocBLAS context.
    //
    // Similar to creating a cuBLAS handle in CUDA.
    // The handle stores library state and execution context.
    //
    rocblas_handle handle;
    rocblas_create_handle(&handle);

    //
    // HIP events are used for accurate GPU timing.
    //
    // CPU timers are often misleading because GPU
    // operations execute asynchronously.
    //
    hipEvent_t t0,t1;
    hipEventCreate(&t0);
    hipEventCreate(&t1);

    //
    // Warm-up run.
    //
    // The first invocation may include:
    //   - ROCm runtime initialization
    //   - kernel loading/JIT compilation
    //   - cache population
    //   - library autotuning
    //
    // We exclude this overhead from measurements.
    //
    rocblas_sgemm(handle,
                  rocblas_operation_none,
                  rocblas_operation_none,
                  M, N, K,
                  &alpha,
                  d_A, M,
                  d_B, K,
                  &beta,
                  d_C, M);

    hipDeviceSynchronize();

    //
    // Benchmark phase.
    //
    // Execute GEMM 20 times and compute
    // the average execution time.
    //
    hipEventRecord(t0);

    for (int i = 0; i < 20; i++)
    {
        rocblas_sgemm(handle,
                      rocblas_operation_none,
                      rocblas_operation_none,
                      M, N, K,
                      &alpha,
                      d_A, M,
                      d_B, K,
                      &beta,
                      d_C, M);
    }

    hipEventRecord(t1);
    hipEventSynchronize(t1);

    float ms;
    hipEventElapsedTime(&ms,t0,t1);

    // Average execution time per GEMM call
    ms /= 20;

    // Copy result matrix back to host memory
    hipMemcpy(h_C, d_C, sC, hipMemcpyDeviceToHost);

    //
    // GEMM performs approximately:
    //
    //     2 * M * N * K
    //
    // floating-point operations.
    //
    // We convert that into GFLOPS
    // (billions of floating-point operations per second).
    //
    double gflops =
        2.0 * M * N * K /
        (ms * 1e-3) /
        1e9;

    printf("=== rocBLAS SGEMM ===\n");
    printf("Size:    %dx%dx%d\n", M, N, K);
    printf("Time:    %.3f ms\n", ms);
    printf("GFLOPS:  %.1f\n", gflops);

    //
    // Cleanup
    //
    rocblas_destroy_handle(handle);

    hipFree(d_A);
    hipFree(d_B);
    hipFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Writing rocblas_gemm.cpp


In [4]:
import subprocess

comp = subprocess.run(['hipcc', '-O3', '-o', 'rocblas_gemm', 'rocblas_gemm.cpp',
                       '-lrocblas'], capture_output=True, text=True)
if comp.returncode != 0:
    print(f"Compile error:\n{comp.stderr}")
else:
    result = subprocess.run(['./rocblas_gemm'], capture_output=True, text=True)
    print(result.stdout)

=== rocBLAS SGEMM ===
Size:    1024x1024x1024
Time:    0.170 ms
GFLOPS:  12603.0



<div style="background:#f4f6f9; border-left:5px solid #64748b; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#334155; font-size:1.05em; margin-bottom:8px;">Understanding GFLOPS</div>
<p>GEMM computes <code>C = α·A·B + β·C</code>. Each output element needs one multiply and one add per element of K, so total work is:</p><p><b>FLOPs = 2 × M × N × K</b></p><p>For M = N = K = 1024 that is about 2.15 billion operations. GFLOPS = FLOPs ÷ execution_time(s) ÷ 1e9 — it measures how efficiently the GPU runs the workload.</p>
</div>

<div style="background:#eef9f1; border-left:5px solid #2f9e6e; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1f7a52; font-size:1.05em; margin-bottom:8px;">💡 Why GEMM matters</div>
<p>Matrix multiply is the workhorse of modern computing: neural-network training and inference, scientific simulation, and graphics all spend much of their time in GEMM. Benchmarking it measures one of the fundamental building blocks of AI hardware.</p>
</div>

## Theoretical Peak Performance and Efficiency

Measuring GFLOPS is useful, but the number alone does not tell the whole story.

To understand whether a kernel performs well, compare the measured performance with the capabilities of the GPU.

### Work Performed

For matrix multiplication (GEMM):

```text
Work = 2 × M × N × K
```

For:

```text
M = N = K = 1024
```

the total work is approximately

```text
2.15 GFLOP
```

### Interpreting Performance

Modern GPUs have finite compute throughput.
However, small GEMM problems often cannot fully utilize all available compute resources.
Measured performance depends on many factors, including:

- Problem size
- Memory hierarchy
- Library implementation
- GPU architecture

As matrix sizes increase, GPU utilization typically improves.

<div style="
background:#fff8e8;
border-left:6px solid #d8c27a;
padding:14px;
border-radius:8px;
margin:15px 0;
">

<b>Key Takeaway</b><br><br>
A lower-than-expected GFLOPS value does not necessarily indicate poor performance.
When evaluating GEMM performance, always consider:
<ul>
<li>Problem size</li>
<li>Measured GFLOPS</li>
<li>The GPU architecture</li>
</ul>
rather than comparing against a single theoretical peak number.

</div>

In [5]:
# Analyze the rocBLAS GEMM result

M = N = K = 1024

# Total floating-point work:
# GEMM performs 2 * M * N * K floating-point operations.

flops = 2 * M * N * K

# Replace with your measured value
measured_gflops = 66202.9

print(f"Matrix size:      {M} x {N}")
print(f"Total work:       {flops/1e9:.2f} GFLOP")
print(f"Measured:         {measured_gflops/1e3:.1f} TFLOPS")
print(f"Execution time:   {flops / (measured_gflops * 1e9) * 1000:.3f} ms")

Matrix size:      1024 x 1024
Total work:       2.15 GFLOP
Measured:         66.2 TFLOPS
Execution time:   0.032 ms


### Interpreting the Result

The measured throughput depends on many factors, including:

- GPU architecture
- Matrix size
- Memory hierarchy
- rocBLAS implementation

For this reason, the measured performance should not be compared directly with a single theoretical peak value.

Instead, compare:

- different matrix sizes;
- different GPUs;
- different library versions.

The primary goal is to understand how efficiently rocBLAS executes the GEMM operation for the selected workload.

## Exercise 2 — MIOpen Convolution with Autotune

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Goal</div>
<p><b>MIOpen</b> is AMD's deep-learning primitive library for ROCm (the counterpart to NVIDIA's cuDNN). Run a convolution and observe how MIOpen searches for the fastest implementation on the first call, then reuses that cached choice on later calls.</p>
</div>

In [ ]:
# MIOpen convolution via PyTorch
#
# Although this example uses PyTorch, the actual convolution
# implementation is executed by the ROCm MIOpen library.
#
# PyTorch
#    ↓
# MIOpen
#    ↓
# HIP Runtime
#    ↓
# AMD GPU
#
# This allows us to observe MIOpen autotuning behavior without
# writing low-level MIOpen API code.

import torch
import torch.nn.functional as F
import time

# Verify that PyTorch can see a ROCm-compatible GPU.
if not torch.cuda.is_available():
    print("ERROR: No GPU available. Check ROCm installation.")
else:
    device = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    #
    # Create a synthetic convolution workload similar to a
    # convolutional neural network (CNN) layer.
    #
    # Input tensor:
    #   Batch size      = 32
    #   Input channels  = 64
    #   Resolution      = 224 x 224
    #
    # Filter tensor:
    #   Output channels = 128
    #   Input channels  = 64
    #   Kernel size     = 3 x 3
    #
    x = torch.randn(32, 64, 224, 224, device=device)
    w = torch.randn(128, 64, 3, 3, device=device)

    #
    # First execution.
    #
    # This call may trigger:
    #   - ROCm runtime initialization
    #   - MIOpen algorithm search
    #   - Autotuning ("find-db" search)
    #   - Cache population
    #
    # As a result, the first run is typically slower than
    # subsequent executions.
    #
    torch.cuda.synchronize()

    t0 = time.perf_counter()

    y = F.conv2d(x, w, padding=1)

    torch.cuda.synchronize()

    first_run = (time.perf_counter() - t0) * 1000

    #
    # Benchmark phase.
    #
    # MIOpen should now have selected an optimized convolution
    # algorithm and cached the result.
    #
    # Subsequent executions typically run much faster because
    # the autotuning process is skipped.
    #
    times = []

    for _ in range(20):

        # Synchronize before timing because GPU execution
        # is asynchronous with respect to the CPU.
        torch.cuda.synchronize()

        t0 = time.perf_counter()

        y = F.conv2d(x, w, padding=1)

        torch.cuda.synchronize()

        times.append((time.perf_counter() - t0) * 1000)

    avg = sum(times) / len(times)

    #
    # Report results.
    #
    # Students should observe that:
    #
    #   First Run  > Cached Runs
    #
    # demonstrating MIOpen autotuning and cache reuse.
    #
    print(f"\n=== MIOpen Convolution (via PyTorch) ===")
    print(f"Input:     32×64×224×224")
    print(f"Filter:    128×64×3×3")

    print(f"First run: {first_run:.1f} ms  (includes autotune)")
    print(f"Cached:    {avg:.1f} ms  (average of 20 runs)")

    print(f"Autotune overhead: {first_run - avg:.1f} ms")

GPU: AMD Radeon Graphics


<div style="background:#fff7ea; border-left:5px solid #e0a020; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#8a5a06; font-size:1.05em; margin-bottom:8px;">⚠️ Autotuning overhead varies</div>
<p>First-run autotuning time is not fixed — it depends on the ROCm version, GPU architecture, and whether the MIOpen cache is already populated. If the cache is warm, the first call may look as fast as later ones. That is expected and demonstrates the caching mechanism.</p>
</div>

## Exercise 3 — rocFFT Round-Trip

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Goal</div>
<p><b>rocFFT</b> is AMD's Fast Fourier Transform library. Perform a forward FFT followed by an inverse FFT and verify the original signal is reconstructed within floating-point rounding error — this validates the FFT path, not the signal itself.</p>
</div>

<div style="background:#f4f6f9; border-left:5px solid #64748b; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#334155; font-size:1.05em; margin-bottom:8px;">Time domain ↔ frequency domain</div>
<p>A forward FFT moves a signal from the time domain to the frequency domain; the inverse FFT moves it back. If both are correct, FFT followed by IFFT returns the original input.</p>
</div>

```text
Time domain  --FFT-->  Frequency domain  --IFFT-->  Time domain
```

In [ ]:
# rocFFT round-trip via PyTorch
#
# Although this example uses the PyTorch FFT API,
# the underlying FFT implementation is provided by
# the ROCm rocFFT library.
#
# PyTorch
#    ↓
# rocFFT
#    ↓
# HIP Runtime
#    ↓
# AMD GPU
#
# This exercise verifies that a forward FFT followed
# by an inverse FFT reconstructs the original signal.

import torch

device = torch.device('cuda')

#
# Create a synthetic test signal.
#
# In a real application this could represent:
#   - Audio samples
#   - Sensor measurements
#   - Radio signals
#   - Scientific data
#
N = 8192

signal = torch.randn(
    N,
    dtype=torch.float32,
    device=device
)

#
# Forward FFT
#
# Converts the signal from the time domain
# into the frequency domain.
#
# Result:
#   spectrum[k]
#
# contains the frequency content of the signal.
#
spectrum = torch.fft.fft(signal)

#
# Inverse FFT
#
# Converts the frequency-domain representation
# back into the original time-domain signal.
#
recovered = torch.fft.ifft(spectrum).real

#
# Numerical accuracy test.
#
# Because floating-point arithmetic is not exact,
# we expect a very small reconstruction error.
#
# A correct FFT implementation should produce
# an error close to machine precision.
#
error = torch.max(
    torch.abs(signal - recovered)
).item()

print("=== rocFFT Round-Trip Test ===")

print(f"Signal length: {N}")

print(f"Max error:     {error:.2e}")

print(
    f"Status:        "
    f"{'PASS' if error < 1e-5 else 'FAIL'}"
)

## Exercise 4 — PyTorch Training on ROCm

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Goal</div>
<p>Combine the pieces into a full workflow: train <b>ResNet-18</b> on the <b>CIFAR-10</b> image dataset for <b>2 epochs</b> and confirm the whole PyTorch + ROCm stack works (data loading, GPU execution, loss, backprop, parameter updates).</p><p style='margin-bottom:0;'><b>Background:</b> an <i>epoch</i> is one full pass over the dataset; ResNet-18 is a standard image classifier; CIFAR-10 has 60,000 labeled images in 10 classes. Under the hood each step uses rocBLAS (matrix multiply), MIOpen (convolutions), and the HIP runtime — PyTorch orchestrates them automatically.</p>
</div>

In [ ]:
# Train ResNet-18 on CIFAR-10 for 2 epochs
#
# This exercise verifies that the complete PyTorch + ROCm
# training pipeline works correctly.
#
# Components involved:
#
#   Dataset loading      (torchvision)
#   ↓
#   Data preprocessing   (transforms)
#   ↓
#   Neural network       (ResNet-18)
#   ↓
#   Loss function
#   ↓
#   Backpropagation
#   ↓
#   Optimizer update
#   ↓
#   AMD GPU via ROCm
#
# Successful execution demonstrates that the entire AI
# software stack is functioning properly.

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import time
import gc
import os, sys
from contextlib import redirect_stdout
import warnings
from pathlib import Path
import tarfile

# Suppress non-critical warnings to keep notebook output clean
warnings.filterwarnings("ignore")

# Run Python garbage collection before starting
gc.collect()

device = torch.device('cuda')

#
# Data preprocessing pipeline.
#
# ToTensor():
#     Converts image pixels into PyTorch tensors.
#
# Normalize():
#     Scales values into a range that is easier for
#     neural networks to learn from.
#
transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

#
# Prepare the CIFAR-10 dataset.
#
# CIFAR-10 contains:
#   - 60,000 color images
#   - 10 object categories
#   - 32x32 pixel resolution
#
# If the dataset is not already present,
# extract it from archive.
#
DATA_ROOT = Path("/workspace/rocm-certification-17072026/data")
ARCHIVE = DATA_ROOT / "cifar-10-python.tar.gz"
EXTRACTED = DATA_ROOT / "cifar-10-batches-py"

if not EXTRACTED.exists():
    print("Extracting CIFAR-10 dataset...")

    if not ARCHIVE.exists():
        raise RuntimeError(
            f"Cannot find {ARCHIVE}"
        )

    with tarfile.open(ARCHIVE, "r:gz") as tar:
        tar.extractall(DATA_ROOT)

    print("Extraction complete.")
else:
    print("Found existing extracted dataset.")

print("Loading CIFAR-10 dataset")
trainset = torchvision.datasets.CIFAR10(
    root=str(DATA_ROOT),
    train=True,
    download=False,
    transform=transform
)

#
# Reset ROCm memory statistics before benchmarking.
#
# This allows us to report memory usage for
# the current training session only.
#
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

#
# DataLoader feeds mini-batches into the model.
#
# batch_size=128:
#     Process 128 images at a time.
#
# shuffle=True:
#     Randomize sample order each epoch.
#
# num_workers=2:
#     Load data in parallel.
#
# pin_memory=True:
#     Speeds up CPU→GPU transfers.
#
loader = torch.utils.data.DataLoader(
    trainset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True)

#
# Create a ResNet-18 model.
#
# ResNet is one of the most influential convolutional
# neural network architectures and is widely used for
# image classification tasks.
#
model = torchvision.models.resnet18(
    num_classes=10
).to(device)

#
# CrossEntropyLoss measures prediction error.
#
# SGD performs parameter updates using gradient descent.
#
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9)

print("\n=== Training ResNet-18 on CIFAR-10 ===")

#
# Training loop.
#
# Each epoch processes the entire dataset once.
#
for epoch in range(2):

    model.train()

    running_loss = 0.0

    # Synchronize before timing because GPU execution
    # is asynchronous with respect to the CPU.
    torch.cuda.synchronize()

    t0 = time.perf_counter()

    for i, (inputs, labels) in enumerate(loader):

        # Move training data to GPU memory
        inputs, labels = inputs.to(device), labels.to(device)

        #
        # Forward pass:
        #   Input → Neural Network → Predictions
        #
        optimizer.zero_grad()

        outputs = model(inputs)

        #
        # Compute prediction error
        #
        loss = criterion(outputs, labels)

        #
        # Backpropagation:
        # Compute gradients for all trainable parameters.
        #
        loss.backward()

        #
        # Update model weights.
        #
        optimizer.step()

        running_loss += loss.item()

        if (i + 1) % 100 == 0:
            print(
                f"  Epoch {epoch+1}, "
                f"Batch {i+1}/{len(loader)}, "
                f"Loss: {running_loss/100:.3f}"
            )

            running_loss = 0.0

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - t0

    print(
        f"  Epoch {epoch+1} complete — "
        f"{elapsed:.1f}s"
    )

print("\n✓ Training complete.")

#
# Report peak GPU memory usage observed during training.
#
print(
    f"  GPU memory used: "
    f"{torch.cuda.max_memory_allocated()/1e9:.2f} GB"
)

<div style="background:#f4f6f9; border-left:5px solid #64748b; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#334155; font-size:1.05em; margin-bottom:8px;">What to observe</div>
<p>The reported loss should generally decrease, indicating the network is learning. Two epochs will not reach high accuracy — the goal here is to confirm the training pipeline executes correctly on the GPU.</p>
</div>

## Exercise 5 — PyTorch Debugging

<div style="background:#f3f0fb; border-left:5px solid #7c5cd6; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#4c3a8c; font-size:1.05em; margin-bottom:8px;">📝 Diagnose and fix</div>
<p>Your instructor will set one of the environment conditions below to simulate a common failure. Use the diagnostic script to identify it, then apply the fix.</p>
</div>

| Failure | Symptom | Fix |
|---------|---------|-----|
| `HIP_VISIBLE_DEVICES=99` | `RuntimeError: No HIP GPUs available` | Set to a valid ID (0) or unset |
| Wrong PyTorch wheel | `torch.version.hip` is `None` | Reinstall the ROCm wheel |
| Missing `--device` flag | GPU not visible in container | Relaunch with correct flags |
| `PYTORCH_ROCM_ARCH` mismatch | Kernel launch failure | Match to the actual GPU arch |

In [ ]:
# Diagnostic script — run this to check your environment
import torch
import os

###########################################
# Set the environment variable with error
#os.environ['HIP_VISIBLE_DEVICES'] = '99'
#The following does not fix the error. 
#JupyterLab kernel reset is required
#os.environ.pop('HIP_VISIBLE_DEVICES', None)
###########################################

print("=== PyTorch ROCm Diagnostics ===")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.version.hip:         {torch.version.hip}")
print(f"HIP_VISIBLE_DEVICES:       {os.environ.get('HIP_VISIBLE_DEVICES', '(not set)')}")
print(f"PYTORCH_ROCM_ARCH:         {os.environ.get('PYTORCH_ROCM_ARCH', '(not set)')}")

if torch.cuda.is_available():
    print(f"Device count:              {torch.cuda.device_count()}")
    print(f"Device name:               {torch.cuda.get_device_name(0)}")
    # Quick compute test
    try:
        x = torch.randn(100, 100, device='cuda')
        y = torch.mm(x, x)
        print(f"Compute test:              PASS")
    except Exception as e:
        print(f"Compute test:              FAIL — {e}")
else:
    print("\n⚠  No GPU detected. Possible causes:")
    print("   1. HIP_VISIBLE_DEVICES set to invalid value")
    print("   2. Container launched without --device=/dev/kfd --device=/dev/dri")
    print("   3. Wrong PyTorch build (CUDA instead of ROCm)")

---

## Lab Submission

| Exercise | Result |
|----------|--------|
| rocBLAS GEMM GFLOPS | |
| rocBLAS GEMM efficiency (% of FP32 peak) | |
| MIOpen first-run time (ms) | |
| MIOpen cached time (ms) | |
| rocFFT round-trip error | |
| PyTorch training loss (epoch 2, final) | |
| Debugging: failure identified | |
| Debugging: fix applied | |

**Next:** Module 4 — HIP Programming & Porting CUDA